para empezar darle a runtime, change runtime type y poner: T4 GPU

01. Configuración del entorno

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers accelerate

In [ ]:
!pip install -q bitsandbytes==0.48.2

In [ ]:
import importlib.metadata

print("bitsandbytes:", importlib.metadata.version("bitsandbytes"))

In [ ]:
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.16.1
Accelerate: 1.14.0


02. Autenticación Hugging Face

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '691518635cb559bdce2dd0fa', 'name': 'MBreadd', 'fullname': 'Saldaña Sánchez Luis Mario', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/196d6e7e55c0cf7cc9f1fc3c0aca02b3.svg', 'orgs': [{'type': 'org', 'id': '68ecc696bf1e3d158e71b7eb', 'name': 'MCP-1st-Birthday', 'fullname': 'MCP-1st-Birthday', 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/60d2dc1007da9c17c72708f8/N6eZhg0R7vFoxsPQODTOI.png'}], 'auth': {'type': 'oauth', 'expiresAt': '2026-10-23T15:52:42.000Z'}}


03. Cargar tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "google/gemma-2-9b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer cargado correctamente")

04. Cargar Gemma 2 9B (8-bit)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

model.eval()

print("Modelo cargado correctamente")

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Modelo cargado correctamente


In [ ]:
print("GPU:", torch.cuda.get_device_name(0))

05. Descargar GSM8K

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")

print(dataset)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [ ]:
print("Train:", len(dataset["train"]))
print("Test:", len(dataset["test"]))

Train: 7473
Test: 1319


In [ ]:
print(dataset["test"][0])

{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [ ]:
gsm8k_test = dataset["test"].select(range(20))

print("Problemas seleccionados:", len(gsm8k_test))

Problemas seleccionados: 20


In [ ]:
for i in range(3):
    print(f"\n--- Problema {i+1} ---")
    print(gsm8k_test[i]["question"])
    print("Respuesta esperada:", gsm8k_test[i]["answer"])


--- Problema 1 ---
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Respuesta esperada: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

--- Problema 2 ---
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Respuesta esperada: It takes 2/2=<<2/2=1>>1 bolt of white fiber
So the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric
#### 3

--- Problema 3 ---
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
Respuesta esperada: The cost of the house and repairs came out to 80,000+50,000=$<<800

06. Cargar perturbaciones

In [ ]:
# Mismo código que src/perturbations.py, reimplementado inline porque este
# notebook corre standalone en Colab (sin el repo clonado / sys.path a ../src).
import json
import random


def perturb_whitespace(prompt: str, seed: int = None) -> str:
    """Inserta espacios dobles y/o un salto de línea final, sin tocar las palabras."""
    if seed is not None:
        random.seed(seed)

    words = prompt.split(" ")
    perturbed_words = []
    for i, word in enumerate(words):
        perturbed_words.append(word)
        if i < len(words) - 1 and random.random() < 0.4:
            perturbed_words.append("")
    perturbed = " ".join(perturbed_words)

    if random.random() < 0.5:
        perturbed += "\n"

    return perturbed


def perturb_markdown(prompt: str, style: str = None, seed: int = None) -> str:
    """Envuelve el prompt en Markdown/XML (bold, header o xml) sin cambiar el texto."""
    styles = {
        "bold": lambda p: f"**Question:** {p}",
        "header": lambda p: f"## Question\n{p}",
        "xml": lambda p: f"<question>{p}</question>",
    }

    if style is None:
        if seed is not None:
            random.seed(seed)
        style = random.choice(list(styles.keys()))

    if style not in styles:
        raise ValueError(f"Unknown style '{style}'. Choose from {list(styles.keys())}")

    return styles[style](prompt)


def perturb_json(prompt: str) -> str:
    """Reestructura el prompt como un objeto JSON {"task", "question"}."""
    payload = {
        "task": "math_problem",
        "question": prompt,
    }
    return json.dumps(payload)


print("Perturbaciones cargadas (whitespace, markdown, json)")

07. Ejecutar experimento

In [ ]:
from tqdm.auto import tqdm

# gemma-2-9b-it espera el formato de chat (<start_of_turn>user ...). Sin
# apply_chat_template seguiríamos con el prompt "crudo" que fue justamente
# el problema confirmado en la celda de prueba (respondía "[Answer 1]\n\n2 +"
# en vez de resolver la pregunta).
MAX_NEW_TOKENS = 200  # bajalo (p.ej. 60) si el tiempo en Colab apremia

raw_results = []  # uno por (problem_id, perturbation_type): logits del primer token + texto generado

for idx in tqdm(range(len(gsm8k_test)), desc="Problemas GSM8k"):
    question = gsm8k_test[idx]["question"]
    expected_answer_raw = gsm8k_test[idx]["answer"]

    variants = {
        "base": question,
        "whitespace": perturb_whitespace(question, seed=1000 + idx),
        "markdown": perturb_markdown(question, seed=1000 + idx),
        "json": perturb_json(question),
    }

    for perturbation_type, variant_text in variants.items():
        chat = [{"role": "user", "content": variant_text}]
        formatted_prompt = tokenizer.apply_chat_template(
            chat, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                output_scores=True,
                return_dict_in_generate=True,
            )

        # Logits del primer token generado (shape [1, vocab_size]) -- el resto
        # de outputs.scores no se usa: la entropía del paper se mide en t=0.
        first_token_logits = outputs.scores[0][0].detach().cpu().float().numpy()

        generated_ids = outputs.sequences[0][inputs["input_ids"].shape[1]:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        raw_results.append({
            "problem_id": idx,
            "perturbation_type": perturbation_type,
            "first_token_logits": first_token_logits,
            "generated_text": generated_text,
            "expected_answer_raw": expected_answer_raw,
        })

print(f"Generación completa: {len(raw_results)} filas ({len(gsm8k_test)} problemas x 4 variantes)")

8. Calcular entropía

In [ ]:
# Mismas fórmulas que src/entropy.py (Ec. 1-2 del README), reimplementadas
# inline por la misma razón que la sección 06.
import numpy as np

def softmax(logits: np.ndarray) -> np.ndarray:
    z = logits - np.max(logits)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

def shannon_entropy(logits: np.ndarray, base: int = 2) -> float:
    probs = softmax(logits)
    probs = probs[probs > 0]  # evita log(0)
    if base == 2:
        return float(-np.sum(probs * np.log2(probs)))
    return float(-np.sum(probs * np.log(probs)))

def delta_entropy(entropy_base: float, entropy_perturbed: float) -> float:
    return entropy_perturbed - entropy_base

for r in raw_results:
    r["H"] = shannon_entropy(r["first_token_logits"])

# H_base por problema, para poder calcular delta_H de cada perturbación
h_base_by_problem = {
    r["problem_id"]: r["H"] for r in raw_results if r["perturbation_type"] == "base"
}

for r in raw_results:
    r["H_base"] = h_base_by_problem[r["problem_id"]]
    r["delta_H"] = delta_entropy(r["H_base"], r["H"])

print(f"Entropía calculada para {len(raw_results)} filas")


09. Guardar resultados

In [ ]:
import os
import re

import matplotlib.pyplot as plt
import pandas as pd

def extract_predicted_number(text: str):
    """Heurística: toma el último número que aparece en el texto generado."""
    matches = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if not matches:
        return None
    return float(matches[-1].replace(",", ""))

def extract_expected_number(answer_raw: str) -> float:
    final = answer_raw.split("####")[-1].strip()
    return float(final.replace(",", ""))

for r in raw_results:
    predicted = extract_predicted_number(r["generated_text"])
    expected = extract_expected_number(r["expected_answer_raw"])
    r["correct"] = predicted is not None and abs(predicted - expected) < 1e-4

# "base" es la referencia para delta_H, no una fila de perturbación en sí
rows = [
    {
        "problem_id": r["problem_id"],
        "perturbation_type": r["perturbation_type"],
        "H_base": r["H_base"],
        "H_perturbed": r["H"],
        "delta_H": r["delta_H"],
        "correct": r["correct"],
    }
    for r in raw_results
    if r["perturbation_type"] != "base"
]

df = pd.DataFrame(rows)
print(df.to_string(index=False))

os.makedirs("results/figures", exist_ok=True)

csv_path = "results/gemma_full_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nGuardado en {csv_path}")

fig, ax = plt.subplots(figsize=(6, 4))
df.boxplot(column="delta_H", by="perturbation_type", ax=ax)
ax.set_title(f"delta_H by perturbation type (gemma-2-9b-it, n={len(gsm8k_test)} problems)")
ax.set_ylabel("delta_H (bits)")
plt.suptitle("")
fig.tight_layout()

fig_path = "results/figures/gemma_entropy_boxplot.png"
fig.savefig(fig_path, dpi=150)
print(f"Guardado en {fig_path}")

contendido de prueba del modelo

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))

prompt = "What is 2 + 2?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

respuesta = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(respuesta)

GPU: Tesla T4
What is 2 + 2?

[Answer 1]

2 + 


In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False,
    output_scores=True,
    return_dict_in_generate=True
)

print("Número de pasos:", len(outputs.scores))
print("Shape del primer score:", outputs.scores[0].shape)
print("Tipo:", outputs.scores[0].dtype)

Número de pasos: 5
Shape del primer score: torch.Size([1, 256000])
Tipo: torch.float32


fin jeje